# Echoes of Astra: voices with ElevenLabs v3

This notebook records every voice line in the game (about 400 clips) with your ElevenLabs account and packs them into a ZIP file. Nothing to install: everything runs in your browser on Google's servers.

**How to use it:**
1. Click ▶ next to each step, **one at a time, top to bottom**. If Colab warns that the notebook was not written by Google, choose **Run anyway**.
2. In step 1, paste your API key into the field that appears. The key stays in this session only and is never saved anywhere.
3. Step 3 designs the unique voices: listen to them. Step 4 plays a short test.
4. Step 5 records everything and takes about 20–40 minutes. If it stops, run step 5 again and it carries on from where it stopped.
5. Step 6 downloads the ZIP (sometimes several parts). Send it to Claude in the chat.
6. Step 7 (optional) removes the designed voices from your account, to free up voice slots.


In [ ]:
#@title 1) API key and account check
import getpass, json, os, time, base64, zipfile, threading, concurrent.futures as cf
import requests
from IPython.display import HTML, display

API = 'https://api.elevenlabs.io'
KEY = getpass.getpass('Paste your ElevenLabs API key and press Enter: ').strip()
S = requests.Session(); S.headers.update({'xi-api-key': KEY})

def call(method, path, *, retries=7, **kw):
    last = None
    for a in range(retries):
        try:
            r = S.request(method, API + path, timeout=300, **kw)
        except requests.RequestException as e:
            last = e; time.sleep(3 + 3 * a); continue
        if r.status_code in (409, 429, 500, 502, 503, 504):
            last = r; time.sleep(min(60, 4 * (a + 1))); continue
        return r
    if isinstance(last, requests.Response): return last
    raise RuntimeError(f'No connection to ElevenLabs: {last}')

def err(r):
    try:
        d = r.json(); return json.dumps(d.get('detail', d), ensure_ascii=False)[:500]
    except Exception:
        return r.text[:500]

def player(b, mime='audio/mpeg', label=''):
    display(HTML(f'<div style="margin:4px 0"><b>{label}</b><br><audio controls src="data:{mime};base64,{base64.b64encode(b).decode()}"></audio></div>'))

rv = call('GET', '/v1/voices')
if rv.status_code != 200:
    raise SystemExit(f'The key does not work ({rv.status_code}): {err(rv)}\nCheck the key and its permissions (Voices: Read/Write, Text to Speech, Text to Voice).')
VOICES = rv.json().get('voices', [])
SUB = {}
rs = call('GET', '/v1/user/subscription')
if rs.status_code == 200:
    SUB = rs.json()
    left = SUB.get('character_limit', 0) - SUB.get('character_count', 0)
    print(f"Plan: {SUB.get('tier')} · credits left this month: {left:,} of {SUB.get('character_limit', 0):,} · custom voice limit: {SUB.get('voice_limit')}")
else:
    print('No access to plan details (the key lacks "User: Read"). Assuming the Starter plan.')
print(f'Voices visible on the account: {len(VOICES)}. Everything OK, go to step 2.')


In [ ]:
#@title 2) Load the script and plan the recording
BIBLE = json.loads(r"""{"designs": [{"key": "dragon", "description": "An ancient, colossal dragon. Extremely deep, resonant, gravelly male voice with enormous chest resonance; slow, regal and contemptuous, with a low growl at the ends of phrases. Dark fantasy video game villain.", "text": "Bow, insects. I have burned kingdoms older than your bloodline, and I will burn yours before the sun has set. Kneel, and perhaps I will let you watch.", "seed": 1101}, {"key": "demon_queen", "description": "A demon queen. Low, velvety, cold female voice with a faint smoky rasp; amused and cruel, speaking slowly with elegant menace. Dark fantasy video game villain.", "text": "Hope is such a fragile little thing. I have watched it flicker out in a thousand eyes, and every time, it was beautiful. Come closer. Let me watch yours.", "seed": 1102}, {"key": "demon_brute", "description": "A hulking demon warrior. Very deep, rough, snarling male voice, heavy breathing, guttural and aggressive, barely controlling his rage. Dark fantasy video game.", "text": "You fed a soul to the pit, and the pit sent me. I will tear down your walls, break your banners, and feast on every scream. Run, if it makes you feel better.", "seed": 1103}, {"key": "mad_scientist", "description": "An excitable, theatrical vampire scientist. Middle-aged British male voice, quick and slightly nasal, giggling between words, delighted by gruesome experiments, a little unhinged.", "text": "Ahh, fresh variables! Hold still, hold still, this will only sting for a moment. Oh, look at that reaction! Fascinating. Somebody write that down, quickly!", "seed": 1104}, {"key": "seraph", "description": "A serene celestial angel. Luminous, airy, warm female voice with gentle authority; calm and soothing, speaking slowly, as if from a vast cathedral.", "text": "The light descends. Rise, fallen ones, for your war is not over. Mercy burns brighter than hate, and no darkness has ever outlasted the dawn.", "seed": 1105}, {"key": "kobold", "description": "A tiny, scrappy lizard creature. High-pitched, raspy, eager young male voice, fast and a little squeaky, overconfident and comically brave. Fantasy video game.", "text": "I'm a dragon too! Well, a small one. Sort of! But my teeth are very sharp, and I bite really, really fast. Fear me! Please? Just a little?", "seed": 1106}, {"key": "combat_ai", "description": "A military combat artificial intelligence. Calm, precise, synthetic female voice, clipped and emotionless, perfectly even pacing, slightly metallic. Science fiction video game.", "text": "Vanguard unit online. All systems green. Hostile signatures detected in sector three. Engaging protocol initiated. Stand clear of the firing line.", "seed": 1107}, {"key": "nature_god", "description": "An ancient primal god of nature. Deep, earthy, slow, rumbling male voice, warm but immense, like wind moving through an old forest. Fantasy video game.", "text": "The wild answers my call. Come forth, children of the earth, root and claw and wing. What was taken from the forest will return to the soil.", "seed": 1108}, {"key": "cat_witch", "description": "A sly feline vampire witch. Playful, sultry young female voice with a purring, teasing, mischievous lilt; she enjoys toying with her prey. Fantasy video game.", "text": "Mmm, the blood moon is purring tonight, and so am I. Bleed for me, darling. Slowly, if you can. I have nine lives, and you have only the one.", "seed": 1109}, {"key": "abomination", "description": "A stitched-together laboratory abomination. Wet, guttural, distorted male voice, slow and labored, groaning between words, barely human. Horror video game monster.", "text": "Consume. Assimilate. Feed the machine. Stitched together, flesh and wire, we grow stronger with every piece you leave behind. Come closer.", "seed": 1110}, {"key": "baroness", "description": "A decadent vampire baroness. Elegant, refined, husky British female voice; languid and seductive, aristocratic, thirsty for blood.", "text": "Kneel, darlings. Your Baroness has arrived, and she is thirsty. Your blood is a fine vintage, and I intend to take the whole bottle.", "seed": 1111}, {"key": "paladin", "description": "A noble holy paladin. Clear, strong, warm female voice, resolute and righteous, ringing with conviction.", "text": "By the Concord's light, judgment comes. Stand before me and be weighed. Kneel before justice, and stay down. The light remains, even when I fall.", "seed": 1112}, {"key": "bunny", "description": "A bubbly rabbit-girl brawler. Bright, energetic young female voice, cocky and playful, giggly, always excited for a fight.", "text": "Brynn's here! Okay, who wants to get smashed first? Bonk! Ha, you didn't even scratch me! Come on, come on, one more round!", "seed": 1113}, {"key": "samurai", "description": "A noble fox samurai. Calm, low, disciplined male voice, measured and honorable, speaking with quiet intensity.", "text": "I am Akeno. Draw your blade, and face me with honor. One cut, one truth. Cherry blossoms fall so beautifully, and so must we all.", "seed": 1114}, {"key": "captain", "description": "A seasoned captain of the guard. Strong, warm baritone male voice, commanding and inspiring, a born battlefield leader.", "text": "Guard, form up! I lead, you follow, and nobody falls behind. With me, strike as one, and strike true! We hold this line until the sun comes up.", "seed": 1115}, {"key": "merchant", "description": "A fast-talking rabbit merchant. Cheerful, slightly nasal, middle-aged male voice with salesman patter, charming and sly.", "text": "Step right up! Something for everyone, and a little extra for you, friend. No refunds, no returns, no exceptions, but oh, what bargains!", "seed": 1116}, {"key": "gladiator", "description": "A massive dragonkin gladiator. Booming, gravelly, showboating male voice, loud and theatrical, always playing to the crowd.", "text": "Step aside! The champion takes the arena! Let me hear you! The crowd wants blood, and I always deliver. Who is next? Who wants to be famous?", "seed": 1117}, {"key": "oracle", "description": "An ancient dragon oracle. Deep, wise, mystical male voice, calm and prophetic, with a soft rumble beneath every word.", "text": "I have seen this battle a thousand times, little one, and you lose in every one. Your future is already ash. I foresaw even this moment.", "seed": 1118}, {"key": "pontiff", "description": "An aging holy emperor and high priest. Grand, sonorous, elderly male voice, solemn and righteous, echoing authority.", "text": "Heaven opens its gates for the faithful. Kneel, and be lifted. Fall before the Direction, heretic, for no sinner escapes the eyes of heaven.", "seed": 1119}, {"key": "scion", "description": "A young vampire heiress of a crime family. Smooth, low, sultry female voice, possessive and dangerous, purring her threats.", "text": "What's yours is mine. It always was. Drown in my shadow, and wake up as mine. Umbrasco never forgets, and Umbrasco never forgives.", "seed": 1120}, {"key": "assassin", "description": "A professional assassin. Quiet, cool, breathy female voice, detached and precise, almost a whisper.", "text": "Target marked. Contract signed. Nothing personal. Quiet now, this will only take a second. You will not even hear me coming.", "seed": 1121}, {"key": "butler", "description": "An impeccably polite vampire butler. Old, dry, refined British male voice, calm and courteous, with dark humor.", "text": "At your service, my liege. Tea, or bloodshed? Do pardon the intrusion, sir. It has been an honor serving this house for three hundred years.", "seed": 1122}, {"key": "giddy_scientist", "description": "A giddy young vampire scientist. Bright, fast, slightly manic female voice, cheerful about terrible experiments.", "text": "Subject Two, wake up! It's testing time, and you're my favorite! For science! Well, mostly for science. Somebody finish my notes, page forty!", "seed": 1123}, {"key": "squire", "description": "An eager young squire. Light, earnest, youthful male voice, nervous but brave, cracking with excitement.", "text": "Hey, ugly! Yes, you! Look at me, not them! For my promotion, and for the banner! I've been practicing all week, teacher, just watch me!", "seed": 1124}, {"key": "inquisitor", "description": "A cold holy inquisitor. Sharp, controlled, icy female voice, precise and intimidating.", "text": "Confess. Your little tricks won't save you now. Heresy is punished, every single time. The Direction sees all, and it does not forgive.", "seed": 1125}, {"key": "firedancer", "description": "An ecstatic fire dancer. Warm, breathy, joyful female voice, wild and rhythmic, laughing with excitement.", "text": "Dance with me in the flames! Don't be shy! Feel the rhythm of fire, now spin, spin, spin! The music never stops while the embers burn.", "seed": 1126}, {"key": "blademaiden", "description": "A graceful fox swordswoman. Smooth, confident, slightly teasing female voice, calm and elegant.", "text": "Nine tails, one blade. Shall we dance? Too slow, darling, far too slow. Every step you take, I have already taken twice.", "seed": 1127}, {"key": "braggart", "description": "A vain dragonkin warrior. Pompous, booming, self-admiring male voice, theatrical and arrogant.", "text": "Behold! The most magnificent warrior you will ever see. Try not to swoon. Witness my greatness, up close, and remember my name forever.", "seed": 1128}], "cast": {"IMP002": {"design": null, "premade": ["Charlie", "Chris", "Eric"], "gender": "male"}, "IMP003": {"design": "cat_witch", "premade": ["Charlotte", "Jessica", "Laura"], "gender": "female"}, "IMP004": {"design": "demon_brute", "premade": ["Callum", "Brian"], "gender": "male"}, "IMP005": {"design": "demon_queen", "premade": ["Aria", "Alice"], "gender": "female"}, "IMP006": {"design": null, "premade": ["Daniel", "George"], "gender": "male"}, "IMP008": {"design": "demon_brute", "premade": ["Brian", "Callum"], "gender": "male"}, "IMP009": {"design": "dragon", "premade": ["Brian", "George"], "gender": "male"}, "IMP010": {"design": "baroness", "premade": ["Lily", "Alice"], "gender": "female"}, "IMP011": {"design": "scion", "premade": ["Charlotte", "Aria"], "gender": "female"}, "IMP012": {"design": "assassin", "premade": ["Sarah", "River"], "gender": "female"}, "IMP013": {"design": null, "premade": ["Brian", "Roger"], "gender": "male"}, "IMP014": {"design": "giddy_scientist", "premade": ["Laura", "Jessica"], "gender": "female"}, "IMP015": {"design": "demon_queen", "premade": ["Aria", "Lily"], "gender": "female"}, "IMP016": {"design": "demon_brute", "premade": ["Daniel", "Callum"], "gender": "male"}, "IMP017": {"design": null, "premade": ["Laura", "Jessica"], "gender": "female"}, "IMP018": {"design": "butler", "premade": ["George", "Daniel"], "gender": "male"}, "IMP019": {"design": "mad_scientist", "premade": ["Callum", "George"], "gender": "male"}, "IMP033": {"design": null, "premade": ["Alice", "Matilda"], "gender": "female"}, "IMP037": {"design": "squire", "premade": ["Liam", "Will"], "gender": "male"}, "IMP039": {"design": null, "premade": ["Daniel", "Roger"], "gender": "male"}, "IMP040": {"design": "bunny", "premade": ["Jessica", "Laura"], "gender": "female"}, "IMP041": {"design": "paladin", "premade": ["Alice", "Matilda"], "gender": "female"}, "IMP042": {"design": null, "premade": ["Eric", "Roger"], "gender": "male"}, "IMP043": {"design": null, "premade": ["Sarah", "River"], "gender": "female"}, "IMP044": {"design": null, "premade": ["Roger", "Chris"], "gender": "male"}, "IMP045": {"design": null, "premade": ["Jessica", "Laura"], "gender": "female"}, "IMP046": {"design": null, "premade": ["Matilda", "Alice"], "gender": "female"}, "IMP047": {"design": null, "premade": ["Sarah", "Aria"], "gender": "female"}, "IMP048": {"design": "seraph", "premade": ["Lily", "Sarah"], "gender": "female"}, "IMP051": {"design": "captain", "premade": ["Roger", "Brian"], "gender": "male"}, "IMP066": {"design": "pontiff", "premade": ["George", "Bill"], "gender": "male"}, "IMP069": {"design": null, "premade": ["Callum", "Liam"], "gender": "male"}, "IMP070": {"design": "inquisitor", "premade": ["Alice", "Aria"], "gender": "female"}, "IMP071": {"design": null, "premade": ["Chris", "Roger"], "gender": "male"}, "IMP072": {"design": null, "premade": ["Lily", "Alice"], "gender": "female"}, "IMP073": {"design": null, "premade": ["Laura", "Jessica"], "gender": "female"}, "IMP075": {"design": null, "premade": ["Alice", "Aria"], "gender": "female"}, "IMP077": {"design": null, "premade": ["Liam", "Will"], "gender": "male"}, "IMP078": {"design": null, "premade": ["Matilda", "Laura"], "gender": "female"}, "IMP080": {"design": "merchant", "premade": ["Charlie", "Eric"], "gender": "male"}, "IMP096": {"design": null, "premade": ["Brian", "Callum"], "gender": "male"}, "IMP097": {"design": null, "premade": ["Laura", "Jessica"], "gender": "female"}, "IMP098": {"design": null, "premade": ["Will", "Liam"], "gender": "male"}, "IMP099": {"design": "assassin", "premade": ["River", "Sarah"], "gender": "female"}, "IMP101": {"design": "firedancer", "premade": ["Jessica", "Charlotte"], "gender": "female"}, "IMP102": {"design": "oracle", "premade": ["George", "Brian"], "gender": "male"}, "IMP103": {"design": "kobold", "premade": ["Will", "Liam"], "gender": "male"}, "IMP104": {"design": null, "premade": ["Jessica", "Laura"], "gender": "female"}, "IMP105": {"design": null, "premade": ["Lily", "Matilda"], "gender": "female"}, "IMP107": {"design": null, "premade": ["Liam", "Charlie"], "gender": "male"}, "IMP108": {"design": "gladiator", "premade": ["Brian", "Roger"], "gender": "male"}, "IMP109": {"design": null, "premade": ["Laura", "Matilda"], "gender": "female"}, "IMP110": {"design": "braggart", "premade": ["Callum", "George"], "gender": "male"}, "IMP111": {"design": "dragon", "premade": ["Brian", "George"], "gender": "male"}, "IMP113": {"design": "dragon", "premade": ["Brian", "Callum"], "gender": "male"}, "IMP114": {"design": null, "premade": ["Daniel", "Roger"], "gender": "male"}, "IMP127": {"design": "samurai", "premade": ["Daniel", "Callum"], "gender": "male"}, "IMP128": {"design": null, "premade": ["River", "Sarah"], "gender": "female"}, "IMP129": {"design": null, "premade": ["George", "Daniel"], "gender": "male"}, "IMP130": {"design": "nature_god", "premade": ["Brian", "Bill"], "gender": "male"}, "IMP131": {"design": null, "premade": ["Lily", "Matilda"], "gender": "female"}, "IMP132": {"design": "demon_queen", "premade": ["River", "Sarah"], "gender": "female"}, "IMP134": {"design": "blademaiden", "premade": ["Charlotte", "Alice"], "gender": "female"}, "IMP135": {"design": null, "premade": ["Callum", "Daniel"], "gender": "male"}, "IMP136": {"design": null, "premade": ["Lily", "River"], "gender": "female"}, "IMP138": {"design": null, "premade": ["Liam", "Eric"], "gender": "male"}, "IMP139": {"design": null, "premade": ["Matilda", "Alice"], "gender": "female"}, "IMP140": {"design": "squire", "premade": ["Will", "Liam"], "gender": "male"}, "IMP141": {"design": null, "premade": ["Bill", "Brian"], "gender": "male"}, "IMP142": {"design": null, "premade": ["Jessica", "Laura"], "gender": "female"}, "IMP143": {"design": null, "premade": ["Jessica", "Sarah"], "gender": "female"}, "IMP144": {"design": null, "premade": ["Charlie", "Callum"], "gender": "male"}, "IMP163": {"design": null, "premade": ["Callum", "Daniel"], "gender": "male"}, "IMP160": {"design": null, "premade": ["Laura", "Matilda"], "gender": "female"}, "IMP034": {"design": "combat_ai", "premade": ["Sarah", "River"], "gender": "female"}, "IMP035": {"design": null, "premade": ["Roger", "Brian"], "gender": "male"}, "IMP036": {"design": null, "premade": ["Chris", "Eric"], "gender": "male"}, "IMP049": {"design": null, "premade": ["Brian", "Roger"], "gender": "male"}, "IMP050": {"design": "combat_ai", "premade": ["River", "Sarah"], "gender": "female"}, "IMP065": {"design": "combat_ai", "premade": ["River", "Sarah"], "gender": "female"}, "IMP067": {"design": null, "premade": ["Daniel", "George"], "gender": "male"}, "IMP068": {"design": "combat_ai", "premade": ["River", "Alice"], "gender": "female"}, "IMP076": {"design": null, "premade": ["Laura", "Jessica"], "gender": "female"}, "IMP079": {"design": null, "premade": ["Brian", "Callum"], "gender": "male"}, "IMP081": {"design": null, "premade": ["Will", "Eric"], "gender": "male"}, "IMP159": {"design": null, "premade": ["Charlie", "Chris"], "gender": "male"}, "TOKEN_DRONE": {"design": "combat_ai", "premade": ["River", "Sarah"], "gender": "female"}, "IMP001": {"design": "abomination", "premade": ["Brian", "Callum"], "gender": "male"}, "IMP074": {"design": "abomination", "premade": ["Brian", "Callum"], "gender": "male"}, "TOKEN_EXPERIMENT": {"design": "abomination", "premade": ["Brian", "Callum"], "gender": "male"}, "SOV_M": {"design": null, "premade": ["George", "Daniel", "Roger"], "gender": "male"}, "SOV_F": {"design": null, "premade": ["Alice", "Lily", "Matilda"], "gender": "female"}, "ANN": {"design": null, "premade": ["Aria", "Alice", "Lily"], "gender": "female"}}, "jobs": [{"stem": "IMP002_play", "cast": "IMP002", "text": "[sly] Fresh stock, barely used. Ask me no questions, and I'll tell you no lies."}, {"stem": "IMP002_attack", "cast": "IMP002", "text": "[menacing] Hold still. You're worth more in one piece."}, {"stem": "IMP002_death", "cast": "IMP002", "text": "[weakly] Make sure... the lab pays my widow..."}, {"stem": "IMP003_play", "cast": "IMP003", "text": "[purring, playful] Hmm... the blood moon is purring tonight. [laughs softly] So am I."}, {"stem": "IMP003_attack", "cast": "IMP003", "text": "[teasing] Bleed for me, darling. [whispers] Slowly, if you can."}, {"stem": "IMP003_death", "cast": "IMP003", "text": "[pained, then amused] Eight lives left... [chuckles weakly] I'll be back before you blink."}, {"stem": "IMP004_play", "cast": "IMP004", "text": "[growling] You fed a soul to the pit... [shouting] and the pit sent me!"}, {"stem": "IMP004_attack", "cast": "IMP004", "text": "[roaring] I'll tear you apart and feast on the screams!"}, {"stem": "IMP004_death", "cast": "IMP004", "text": "[gasping, rasping] More... I still want... more..."}, {"stem": "IMP005_play", "cast": "IMP005", "text": "[cold, menacing] You paid with a card. I'll collect the rest in blood."}, {"stem": "IMP005_attack", "cast": "IMP005", "text": "[calm, cruel] Kneel. My scythe prefers a clean cut."}, {"stem": "IMP005_death", "cast": "IMP005", "text": "[surprised, fading] Even the reaper... gets reaped..."}, {"stem": "IMP006_play", "cast": "IMP006", "text": "[grandly] Death is a door, and I hold the key. [commanding] Rise, my faithful!"}, {"stem": "IMP006_attack", "cast": "IMP006", "text": "[venomous] Wither, and serve me in the next life!"}, {"stem": "IMP006_death", "cast": "IMP006", "text": "[weak laugh] I'll be back... I always come back..."}, {"stem": "IMP008_play", "cast": "IMP008", "text": "[menacing growl] The Dominion sends its regards... in fire."}, {"stem": "IMP008_attack", "cast": "IMP008", "text": "[shouting] Burn, and watch your victory turn to ash!"}, {"stem": "IMP008_death", "cast": "IMP008", "text": "[fading, rasping] The fire... goes out..."}, {"stem": "IMP009_play", "cast": "IMP009", "text": "[mocking] You buried me? How adorable. [menacing] The grave cannot hold a dragon."}, {"stem": "IMP009_attack", "cast": "IMP009", "text": "[roaring] Feel the cold of a thousand graves!"}, {"stem": "IMP009_death", "cast": "IMP009", "text": "[weakened, defiant] I have died before... I will rise again."}, {"stem": "IMP010_play", "cast": "IMP010", "text": "[languid, aristocratic] Kneel, darlings. Your Baroness has arrived, [sultry] and she is thirsty."}, {"stem": "IMP010_attack", "cast": "IMP010", "text": "[savoring] Your blood is a fine vintage. I'll take the whole bottle."}, {"stem": "IMP010_death", "cast": "IMP010", "text": "[gasping, proud] The house... will remember my name..."}, {"stem": "IMP011_play", "cast": "IMP011", "text": "[smug, possessive] What's yours is mine. It always was. [cold] Kneel before Umbrasco."}, {"stem": "IMP011_attack", "cast": "IMP011", "text": "[whispers, threatening] Drown in my shadow, and wake up as mine."}, {"stem": "IMP011_death", "cast": "IMP011", "text": "[bitter, fading] Umbrasco never forgets... and never forgives..."}, {"stem": "IMP012_play", "cast": "IMP012", "text": "[quiet, professional] Target marked. Contract signed. Nothing personal."}, {"stem": "IMP012_attack", "cast": "IMP012", "text": "[whispers] Quiet now. This will only take a second."}, {"stem": "IMP012_death", "cast": "IMP012", "text": "[pained whisper] Tell the client... I'm sorry..."}, {"stem": "IMP013_play", "cast": "IMP013", "text": "[menacing, casual] Nice little army you've got. Shame if something happened to it."}, {"stem": "IMP013_attack", "cast": "IMP013", "text": "[threatening] Pay up, or bleed. Your choice, friend."}, {"stem": "IMP013_death", "cast": "IMP013", "text": "[gasping] Tell the boss... I didn't squeal..."}, {"stem": "IMP014_play", "cast": "IMP014", "text": "[excited, giddy] Subject Two, wake up! It's testing time, and you're my favorite!"}, {"stem": "IMP014_attack", "cast": "IMP014", "text": "[cheerful] For science! [laughs] Well... mostly for science!"}, {"stem": "IMP014_death", "cast": "IMP014", "text": "[weakly, worried] Somebody... finish my notes... page forty..."}, {"stem": "IMP015_play", "cast": "IMP015", "text": "[slow, velvety menace] Hope is a candle, and I am the wind. [whispers] Abandon it."}, {"stem": "IMP015_attack", "cast": "IMP015", "text": "[amused, cruel] Struggle, little one. Despair tastes sweeter when you fight."}, {"stem": "IMP015_death", "cast": "IMP015", "text": "[cold laugh, fading] You call this hope? It is only... a delay..."}, {"stem": "IMP016_play", "cast": "IMP016", "text": "[barking orders] Comrades! Your lives belong to the Dominion. Spend them well."}, {"stem": "IMP016_attack", "cast": "IMP016", "text": "[shouting] Forward! Retreat is treason!"}, {"stem": "IMP016_death", "cast": "IMP016", "text": "[furious, gasping] Traitors... every last one of you..."}, {"stem": "IMP017_play", "cast": "IMP017", "text": "[fanatical, shouting] For the Dominion, I give everything! Even my last breath!"}, {"stem": "IMP017_attack", "cast": "IMP017", "text": "[ecstatic shout] Glory to Valrath! Glory unending!"}, {"stem": "IMP017_death", "cast": "IMP017", "text": "[gasping, fervent] My death... still serves..."}, {"stem": "IMP018_play", "cast": "IMP018", "text": "[politely, dry] At your service, my liege. Tea, or bloodshed?"}, {"stem": "IMP018_attack", "cast": "IMP018", "text": "[courteous] Do pardon the intrusion."}, {"stem": "IMP018_death", "cast": "IMP018", "text": "[weakly, dignified] It was an honor... master..."}, {"stem": "IMP019_play", "cast": "IMP019", "text": "[giddy, manic] Ahh, fresh variables! [giggles] Let's see what happens if I poke this."}, {"stem": "IMP019_attack", "cast": "IMP019", "text": "[excited] Hold still, specimen! Science waits for no one!"}, {"stem": "IMP019_death", "cast": "IMP019", "text": "[fascinated, fading] Fascinating... so this is what dying feels like..."}, {"stem": "IMP033_play", "cast": "IMP033", "text": "[determined, shouting] Shields up! Nobody gets past me. Nobody!"}, {"stem": "IMP033_attack", "cast": "IMP033", "text": "[strained effort] Shield wall, push! Make them eat the steel!"}, {"stem": "IMP033_death", "cast": "IMP033", "text": "[pained] Shield's broken... hold the line without me..."}, {"stem": "IMP037_play", "cast": "IMP037", "text": "[nervous bravado, shouting] Hey, ugly! Yes, you! Look at me, not them!"}, {"stem": "IMP037_attack", "cast": "IMP037", "text": "[excited] For my promotion! And for the banner!"}, {"stem": "IMP037_death", "cast": "IMP037", "text": "[weak, hopeful] Was I brave enough... to get promoted?"}, {"stem": "IMP039_play", "cast": "IMP039", "text": "[calm, confident] Come, then. My guard has never fallen. Not once."}, {"stem": "IMP039_attack", "cast": "IMP039", "text": "[fierce] Steel answers steel, and mine answers first!"}, {"stem": "IMP039_death", "cast": "IMP039", "text": "[solemn, fading] My oath... outlives me..."}, {"stem": "IMP040_play", "cast": "IMP040", "text": "[excited, cocky] Brynn's here! Okay, who wants to get smashed first?"}, {"stem": "IMP040_attack", "cast": "IMP040", "text": "[gleeful] Bonk! [laughs] Ha! You didn't even scratch me!"}, {"stem": "IMP040_death", "cast": "IMP040", "text": "[whiny, pouting] No fair... I was totally winning..."}, {"stem": "IMP041_play", "cast": "IMP041", "text": "[resolute, ringing] By the Concord's light, judgment comes. Are you worthy?"}, {"stem": "IMP041_attack", "cast": "IMP041", "text": "[stern] Kneel before justice, and stay down."}, {"stem": "IMP041_death", "cast": "IMP041", "text": "[weak but serene] The light remains... even when I fall..."}, {"stem": "IMP042_play", "cast": "IMP042", "text": "[focused, brisk] Recalculating. You there, back on your feet. We're not done yet."}, {"stem": "IMP042_attack", "cast": "IMP042", "text": "[confident] Execute plan B. There's always a plan B."}, {"stem": "IMP042_death", "cast": "IMP042", "text": "[surprised, weak] I didn't... plan for this one..."}, {"stem": "IMP043_play", "cast": "IMP043", "text": "[calm, quiet] Target acquired. Wind is calm. Taking the shot."}, {"stem": "IMP043_attack", "cast": "IMP043", "text": "[exhales slowly] One breath... one shot."}, {"stem": "IMP043_death", "cast": "IMP043", "text": "[pained whisper] Lost... my sightline..."}, {"stem": "IMP044_play", "cast": "IMP044", "text": "[commanding] Drone, on me! Squad, we move out on my mark!"}, {"stem": "IMP044_attack", "cast": "IMP044", "text": "[shouting] Advance! The Republic is watching!"}, {"stem": "IMP044_death", "cast": "IMP044", "text": "[gasping] Keep pushing... forward..."}, {"stem": "IMP045_play", "cast": "IMP045", "text": "[cheerful, hurried] Medic here! Nyaa, hold still, this won't sting much!"}, {"stem": "IMP045_attack", "cast": "IMP045", "text": "[apologetic, playful] Sorry, nyaa! Doctor's orders!"}, {"stem": "IMP045_death", "cast": "IMP045", "text": "[weakly] Who heals... the healer?"}, {"stem": "IMP046_play", "cast": "IMP046", "text": "[inspiring, rousing] Together, we are unbreakable! Stand with me!"}, {"stem": "IMP046_attack", "cast": "IMP046", "text": "[shouting] Every voice, one blade! Now!"}, {"stem": "IMP046_death", "cast": "IMP046", "text": "[tender, fading] Stay united... promise me..."}, {"stem": "IMP047_play", "cast": "IMP047", "text": "[steady, professional] Sentinel reporting. Holding this position until relieved."}, {"stem": "IMP047_attack", "cast": "IMP047", "text": "[shouting] Contact front! Opening fire!"}, {"stem": "IMP047_death", "cast": "IMP047", "text": "[pained] Trooper down... tell my squad..."}, {"stem": "IMP048_play", "cast": "IMP048", "text": "[serene, luminous] The light descends. Rise, fallen ones. Your war is not over."}, {"stem": "IMP048_attack", "cast": "IMP048", "text": "[gentle but powerful] Be cleansed. Mercy burns brighter than hate."}, {"stem": "IMP048_death", "cast": "IMP048", "text": "[calm, peaceful] Light never dies... it only returns."}, {"stem": "IMP051_play", "cast": "IMP051", "text": "[commanding, rousing] Guard, form up! Vero leads, you follow, and nobody falls behind!"}, {"stem": "IMP051_attack", "cast": "IMP051", "text": "[battle cry] With me, Guard! Strike as one, and strike true!"}, {"stem": "IMP051_death", "cast": "IMP051", "text": "[strained, fading] Guard... carry on without me..."}, {"stem": "IMP066_play", "cast": "IMP066", "text": "[solemn, grand] Heaven opens its gates for the faithful. Kneel, and be lifted."}, {"stem": "IMP066_attack", "cast": "IMP066", "text": "[righteous anger] Fall before the Direction, heretic."}, {"stem": "IMP066_death", "cast": "IMP066", "text": "[weak, disbelieving] Heaven... will choose another..."}, {"stem": "IMP069_play", "cast": "IMP069", "text": "[rallying shout] Riders of heaven, descend! Wings out, lances down!"}, {"stem": "IMP069_attack", "cast": "IMP069", "text": "[exhilarated] Strike and soar! They'll never touch us!"}, {"stem": "IMP069_death", "cast": "IMP069", "text": "[gasping] My wings... falter..."}, {"stem": "IMP070_play", "cast": "IMP070", "text": "[cold, icy] Confess. Your little tricks won't save you now."}, {"stem": "IMP070_attack", "cast": "IMP070", "text": "[sharp] Heresy is punished. Every single time."}, {"stem": "IMP070_death", "cast": "IMP070", "text": "[cold, fading] The Direction... still sees you..."}, {"stem": "IMP071_play", "cast": "IMP071", "text": "[authoritative, bored] You're under lockdown. Stay put, and nobody gets hurt."}, {"stem": "IMP071_attack", "cast": "IMP071", "text": "[sarcastic] Resisting arrest? That's another charge."}, {"stem": "IMP071_death", "cast": "IMP071", "text": "[groaning] Lockdown... lifted..."}, {"stem": "IMP072_play", "cast": "IMP072", "text": "[calm, smug] Every piece in its proper place. Including you."}, {"stem": "IMP072_attack", "cast": "IMP072", "text": "[quietly amused] Checkmate approaches. You just can't see it yet."}, {"stem": "IMP072_death", "cast": "IMP072", "text": "[surprised, gracious] A move I didn't foresee... well played..."}, {"stem": "IMP073_play", "cast": "IMP073", "text": "[excited, shouting over engines] Engines hot! Come on, babies, let's show them how we fly!"}, {"stem": "IMP073_attack", "cast": "IMP073", "text": "[thrilled] Full throttle! Go, go, go!"}, {"stem": "IMP073_death", "cast": "IMP073", "text": "[weak, wistful] Keep flying... without me..."}, {"stem": "IMP075_play", "cast": "IMP075", "text": "[stern] None pass without judgment. Least of all you."}, {"stem": "IMP075_attack", "cast": "IMP075", "text": "[fierce] From above! You should have looked up."}, {"stem": "IMP075_death", "cast": "IMP075", "text": "[resigned] Grounded... at last..."}, {"stem": "IMP077_play", "cast": "IMP077", "text": "[manic, euphoric] Synthblood's flowing! [laughs] Ha! I feel alive! I feel everything!"}, {"stem": "IMP077_attack", "cast": "IMP077", "text": "[desperate, frenzied] More! Give me more!"}, {"stem": "IMP077_death", "cast": "IMP077", "text": "[trembling, desperate] Just... one more dose..."}, {"stem": "IMP078_play", "cast": "IMP078", "text": "[bold, excited] Dragoon Valeria, diving in! Clear the skies!"}, {"stem": "IMP078_attack", "cast": "IMP078", "text": "[fierce] Out of my sky, and stay out!"}, {"stem": "IMP078_death", "cast": "IMP078", "text": "[pained, sad] Clipped... wings..."}, {"stem": "IMP080_play", "cast": "IMP080", "text": "[salesman, cheerful] Step right up! Something for everyone, and a little extra for you!"}, {"stem": "IMP080_attack", "cast": "IMP080", "text": "[indignant] No refunds! No returns! No exceptions!"}, {"stem": "IMP080_death", "cast": "IMP080", "text": "[sighs, weakly] Business... is closed..."}, {"stem": "IMP096_play", "cast": "IMP096", "text": "[smug, growling] Nice supplies you've got there. Ours now."}, {"stem": "IMP096_attack", "cast": "IMP096", "text": "[snarling] Pay the toll, or we take it in teeth!"}, {"stem": "IMP096_death", "cast": "IMP096", "text": "[growling, fading] The pack... will remember..."}, {"stem": "IMP097_play", "cast": "IMP097", "text": "[wild, excited] Blitz! Hit them where they're empty, and don't look back!"}, {"stem": "IMP097_attack", "cast": "IMP097", "text": "[whooping] Raid's on! Grab what you can!"}, {"stem": "IMP097_death", "cast": "IMP097", "text": "[gasping] Raid's... over..."}, {"stem": "IMP098_play", "cast": "IMP098", "text": "[sniffing, sly] Scouting ahead. Smells like loot... and trouble."}, {"stem": "IMP098_attack", "cast": "IMP098", "text": "[quick, playful] Quick bite, quick exit!"}, {"stem": "IMP098_death", "cast": "IMP098", "text": "[weak] Lost... the trail..."}, {"stem": "IMP099_play", "cast": "IMP099", "text": "[whispers] The hunt begins in silence. You won't even hear me."}, {"stem": "IMP099_attack", "cast": "IMP099", "text": "[cold whisper] Gone before you bleed."}, {"stem": "IMP099_death", "cast": "IMP099", "text": "[pained, ironic] The hunter... becomes the prey..."}, {"stem": "IMP101_play", "cast": "IMP101", "text": "[laughing, joyful] Dance with me in the flames! Don't be shy!"}, {"stem": "IMP101_attack", "cast": "IMP101", "text": "[ecstatic] Feel the rhythm of fire! Now spin!"}, {"stem": "IMP101_death", "cast": "IMP101", "text": "[sad, fading] The music... stops..."}, {"stem": "IMP102_play", "cast": "IMP102", "text": "[calm, prophetic] I have seen this battle a thousand times. You lose in every one."}, {"stem": "IMP102_attack", "cast": "IMP102", "text": "[booming] Your future is already ash!"}, {"stem": "IMP102_death", "cast": "IMP102", "text": "[serene, fading] I foresaw... even this ending..."}, {"stem": "IMP103_play", "cast": "IMP103", "text": "[proud, squeaky] I'm a dragon too! [nervous] Well... a small one. Sort of!"}, {"stem": "IMP103_attack", "cast": "IMP103", "text": "[fierce but tiny] Bite! Bite! Fear my tiny teeth!"}, {"stem": "IMP103_death", "cast": "IMP103", "text": "[sad, squeaky] Too small... still..."}, {"stem": "IMP104_play", "cast": "IMP104", "text": "[laughing, cheeky] Last one there skips dinner! Ha!"}, {"stem": "IMP104_attack", "cast": "IMP104", "text": "[teasing] Too slow! Way too slow!"}, {"stem": "IMP104_death", "cast": "IMP104", "text": "[annoyed, groaning] Tripped... seriously?"}, {"stem": "IMP105_play", "cast": "IMP105", "text": "[serene, mystical] A hidden ally, foretold in gold. The mountain remembers."}, {"stem": "IMP105_attack", "cast": "IMP105", "text": "[calm certainty] Prophecy fulfilled, as it was written."}, {"stem": "IMP105_death", "cast": "IMP105", "text": "[peaceful, fading] Another dawn... will come..."}, {"stem": "IMP107_play", "cast": "IMP107", "text": "[hyped, shouting] Glory or nothing! Let's go, let's go!"}, {"stem": "IMP107_attack", "cast": "IMP107", "text": "[grunting with effort] One! Two! Three swings, and you're down!"}, {"stem": "IMP107_death", "cast": "IMP107", "text": "[satisfied, fading] Worth... it..."}, {"stem": "IMP108_play", "cast": "IMP108", "text": "[booming showman] Step aside! The champion takes the arena! Let me hear you!"}, {"stem": "IMP108_attack", "cast": "IMP108", "text": "[roaring] The crowd wants blood, and I deliver!"}, {"stem": "IMP108_death", "cast": "IMP108", "text": "[stunned, fading] The crowd... goes quiet..."}, {"stem": "IMP109_play", "cast": "IMP109", "text": "[giddy] Ancient ruins mean ancient treasure. Dibs on everything!"}, {"stem": "IMP109_attack", "cast": "IMP109", "text": "[cheeky] Finders keepers, losers weepers!"}, {"stem": "IMP109_death", "cast": "IMP109", "text": "[heartbroken] My... treasure..."}, {"stem": "IMP110_play", "cast": "IMP110", "text": "[pompous] Behold! The most magnificent warrior you will ever see. Try not to swoon."}, {"stem": "IMP110_attack", "cast": "IMP110", "text": "[arrogant shout] Witness my greatness, up close!"}, {"stem": "IMP110_death", "cast": "IMP110", "text": "[dramatic, fading] Remember... my name..."}, {"stem": "IMP111_play", "cast": "IMP111", "text": "[contemptuous, booming] Bow, insects. The Skyscourge has come, and the sky is mine."}, {"stem": "IMP111_attack", "cast": "IMP111", "text": "[roaring] Your lives are fuel, and I am the fire!"}, {"stem": "IMP111_death", "cast": "IMP111", "text": "[mournful rumble] The sky... weeps..."}, {"stem": "IMP113_play", "cast": "IMP113", "text": "[roaring] Wyrmscar flames awaken!"}, {"stem": "IMP113_attack", "cast": "IMP113", "text": "[roaring] Burn to cinders!"}, {"stem": "IMP114_play", "cast": "IMP114", "text": "[calm, steadfast] The Ember-Warden stands ready. Always ready."}, {"stem": "IMP114_attack", "cast": "IMP114", "text": "[battle cry] Spear of embers, strike true!"}, {"stem": "IMP114_death", "cast": "IMP114", "text": "[solemn, fading] My watch... is ended..."}, {"stem": "IMP127_play", "cast": "IMP127", "text": "[calm, honorable] I am Akeno. Draw your blade, and face me with honor."}, {"stem": "IMP127_attack", "cast": "IMP127", "text": "[sharp, focused] One cut. One truth."}, {"stem": "IMP127_death", "cast": "IMP127", "text": "[peaceful, fading] Cherry blossoms... fall so beautifully..."}, {"stem": "IMP128_play", "cast": "IMP128", "text": "[whispers, amused] Watching from the branches. You never once looked up."}, {"stem": "IMP128_attack", "cast": "IMP128", "text": "[playful whisper] Surprise. From the leaves."}, {"stem": "IMP128_death", "cast": "IMP128", "text": "[weak] Fallen... from the canopy..."}, {"stem": "IMP129_play", "cast": "IMP129", "text": "[solemn, resolute] Elorhain's roots will not be defiled. Not while I stand."}, {"stem": "IMP129_attack", "cast": "IMP129", "text": "[battle cry] For the Ancestral Crown!"}, {"stem": "IMP129_death", "cast": "IMP129", "text": "[fading] The roots... endure..."}, {"stem": "IMP130_play", "cast": "IMP130", "text": "[deep, resonant] The wild answers my call. [commanding] Come forth, children of the earth!"}, {"stem": "IMP130_attack", "cast": "IMP130", "text": "[booming] Return to the earth!"}, {"stem": "IMP130_death", "cast": "IMP130", "text": "[peaceful rumble] I return... to the soil..."}, {"stem": "IMP131_play", "cast": "IMP131", "text": "[mystical, soft] Spirits of the grove, guide my hand. Show me who comes next."}, {"stem": "IMP131_attack", "cast": "IMP131", "text": "[calm, ominous] Nature's wrath is patient... until it isn't."}, {"stem": "IMP131_death", "cast": "IMP131", "text": "[peaceful] Back... to the grove..."}, {"stem": "IMP132_play", "cast": "IMP132", "text": "[eerie whisper] Come closer... feed the roots..."}, {"stem": "IMP132_death", "cast": "IMP132", "text": "[hissing whisper] Wither... rot..."}, {"stem": "IMP134_play", "cast": "IMP134", "text": "[confident, teasing] Nine tails, one blade. Shall we dance?"}, {"stem": "IMP134_attack", "cast": "IMP134", "text": "[amused] Too slow, darling."}, {"stem": "IMP134_death", "cast": "IMP134", "text": "[soft, fading] My tails... grow dim..."}, {"stem": "IMP135_play", "cast": "IMP135", "text": "[quiet, smug] You never saw me coming. No one ever does."}, {"stem": "IMP135_attack", "cast": "IMP135", "text": "[whispers] Behind you. Always behind you."}, {"stem": "IMP135_death", "cast": "IMP135", "text": "[fading whisper] Unseen... to the very end..."}, {"stem": "IMP136_play", "cast": "IMP136", "text": "[dreamy, soft] The moon whispers, and I listen. [whispers] Tonight, it whispers your name."}, {"stem": "IMP136_attack", "cast": "IMP136", "text": "[quiet, focused] Strike, then vanish."}, {"stem": "IMP136_death", "cast": "IMP136", "text": "[soft, fading] The moon... sets..."}, {"stem": "IMP138_play", "cast": "IMP138", "text": "[focused, calm] Bowstring's taut. Just say the word."}, {"stem": "IMP138_attack", "cast": "IMP138", "text": "[sharp] Loose! And another!"}, {"stem": "IMP138_death", "cast": "IMP138", "text": "[pained] Out of... arrows..."}, {"stem": "IMP139_play", "cast": "IMP139", "text": "[rallying shout] Sound the horn! Sylvans, to me! To me!"}, {"stem": "IMP139_attack", "cast": "IMP139", "text": "[energetic] Rally and strike! Together!"}, {"stem": "IMP139_death", "cast": "IMP139", "text": "[weak, urgent] Keep the call... alive..."}, {"stem": "IMP140_play", "cast": "IMP140", "text": "[eager, nervous] I'll do my best, teacher! I've been practicing all week!"}, {"stem": "IMP140_attack", "cast": "IMP140", "text": "[effort shout] Hyah! Just like I practiced!"}, {"stem": "IMP140_death", "cast": "IMP140", "text": "[weak, regretful] Should have... trained more..."}, {"stem": "IMP141_play", "cast": "IMP141", "text": "[gentle, deep] This land is under my protection. Walk softly."}, {"stem": "IMP141_attack", "cast": "IMP141", "text": "[firm, rumbling] Leave our home. Now."}, {"stem": "IMP141_death", "cast": "IMP141", "text": "[sorrowful] Forgive me... gentle forest..."}, {"stem": "IMP142_play", "cast": "IMP142", "text": "[playful, shouting] Hey! Over here! Catch me if you can!"}, {"stem": "IMP142_attack", "cast": "IMP142", "text": "[laughing] Got your attention? Good!"}, {"stem": "IMP142_death", "cast": "IMP142", "text": "[sheepish, weak] Okay... they caught me..."}, {"stem": "IMP143_play", "cast": "IMP143", "text": "[eager, cute] Even a little fox can help! Watch me!"}, {"stem": "IMP143_attack", "cast": "IMP143", "text": "[spirited] Take that! And that!"}, {"stem": "IMP143_death", "cast": "IMP143", "text": "[weak, proud] I did... my part..."}, {"stem": "IMP144_play", "cast": "IMP144", "text": "[mischievous chuckle] Now, now... march where I tell you, puppet."}, {"stem": "IMP144_attack", "cast": "IMP144", "text": "[teasing] Was that real? Are you sure?"}, {"stem": "IMP144_death", "cast": "IMP144", "text": "[fading laugh] Just... an illusion..."}, {"stem": "IMP163_play", "cast": "IMP163", "text": "[quiet, confident] From the veil, a single arrow. That's all it takes."}, {"stem": "IMP163_attack", "cast": "IMP163", "text": "[whispers] Right between the plates."}, {"stem": "IMP163_death", "cast": "IMP163", "text": "[fading] Back... into the shadow..."}, {"stem": "IMP160_play", "cast": "IMP160", "text": "[cheerful, hurried] Special delivery! Sign here, please, and mind the box!"}, {"stem": "IMP160_attack", "cast": "IMP160", "text": "[startled] Uh... return to sender!"}, {"stem": "IMP160_death", "cast": "IMP160", "text": "[sighs, weak] Delivery... late..."}, {"stem": "IMP034_play", "cast": "IMP034", "text": "[calm, synthetic] Guardian online. Go ahead, target me. I dare you."}, {"stem": "IMP034_attack", "cast": "IMP034", "text": "[flat] Engaging target."}, {"stem": "IMP034_death", "cast": "IMP034", "text": "[glitching, slowing down] Hull breach... shutting down..."}, {"stem": "IMP035_play", "cast": "IMP035", "text": "[confident, over the radio] Argentum rolling! Give me your best shot."}, {"stem": "IMP035_attack", "cast": "IMP035", "text": "[shouting] Main cannon, fire!"}, {"stem": "IMP035_death", "cast": "IMP035", "text": "[panicked shouting] Armor's gone! Abandon the tank!"}, {"stem": "IMP036_play", "cast": "IMP036", "text": "[steady, over the radio] Carrier deployed. Payload secure."}, {"stem": "IMP036_attack", "cast": "IMP036", "text": "[shouting] Ramming speed!"}, {"stem": "IMP036_death", "cast": "IMP036", "text": "[urgent] Drones, deploy! Deploy!"}, {"stem": "IMP049_play", "cast": "IMP049", "text": "[professional, over the radio] Howitzer locked and loaded. Coordinates confirmed."}, {"stem": "IMP049_attack", "cast": "IMP049", "text": "[shouting] Fire for effect!"}, {"stem": "IMP049_death", "cast": "IMP049", "text": "[coughing] Barrel's cracked... we're out..."}, {"stem": "IMP050_play", "cast": "IMP050", "text": "[synthetic, even] Vanguard Mech online. All systems green."}, {"stem": "IMP050_attack", "cast": "IMP050", "text": "[cold, synthetic] Stomp protocol engaged."}, {"stem": "IMP050_death", "cast": "IMP050", "text": "[glitching, slowing down] Critical failure..."}, {"stem": "IMP065_play", "cast": "IMP065", "text": "[synthetic, cheerful] Courier online. Package ready for delivery."}, {"stem": "IMP065_attack", "cast": "IMP065", "text": "[synthetic] Delivery incoming."}, {"stem": "IMP065_death", "cast": "IMP065", "text": "[synthetic] Package delivered. Signing off."}, {"stem": "IMP067_play", "cast": "IMP067", "text": "[authoritative, over the radio] Skyfortress in position. Airspace locked down."}, {"stem": "IMP067_attack", "cast": "IMP067", "text": "[commanding shout] All batteries, open fire!"}, {"stem": "IMP067_death", "cast": "IMP067", "text": "[panicked] We're going down! Abandon ship!"}, {"stem": "IMP068_play", "cast": "IMP068", "text": "[synthetic, scanning] Scanning... hidden asset located."}, {"stem": "IMP068_attack", "cast": "IMP068", "text": "[cold, synthetic] Heretic detected."}, {"stem": "IMP068_death", "cast": "IMP068", "text": "[glitching] Scan... terminated..."}, {"stem": "IMP076_play", "cast": "IMP076", "text": "[cocky, over the radio] Gunship inbound, weapons hot."}, {"stem": "IMP076_attack", "cast": "IMP076", "text": "[excited shouting] Strafing run! Hang on!"}, {"stem": "IMP076_death", "cast": "IMP076", "text": "[panicked] Mayday, mayday! We're hit!"}, {"stem": "IMP079_play", "cast": "IMP079", "text": "[cold, booming] Dreadnought online. All targets acquired."}, {"stem": "IMP079_attack", "cast": "IMP079", "text": "[menacing] Annihilation cannon, charging!"}, {"stem": "IMP079_death", "cast": "IMP079", "text": "[alarmed] Reactor critical... total loss..."}, {"stem": "IMP081_play", "cast": "IMP081", "text": "[friendly, over the radio] Transport landing. Your cargo is coming home."}, {"stem": "IMP081_attack", "cast": "IMP081", "text": "[shouting] Shields forward!"}, {"stem": "IMP081_death", "cast": "IMP081", "text": "[dismayed] Cargo... lost..."}, {"stem": "IMP159_play", "cast": "IMP159", "text": "[cheerful, gruff] Excavator online! Let's dig!"}, {"stem": "IMP159_attack", "cast": "IMP159", "text": "[shouting] Clearing the site, stand back!"}, {"stem": "IMP159_death", "cast": "IMP159", "text": "[sputtering] Engine... stalled..."}, {"stem": "TOKEN_DRONE_play", "cast": "TOKEN_DRONE", "text": "[synthetic] Drone active."}, {"stem": "TOKEN_DRONE_attack", "cast": "TOKEN_DRONE", "text": "[synthetic] Engaging."}, {"stem": "TOKEN_DRONE_death", "cast": "TOKEN_DRONE", "text": "[glitching] Signal lost."}, {"stem": "IMP001_play", "cast": "IMP001", "text": "[guttural, groaning] Consume... assimilate!"}, {"stem": "IMP001_attack", "cast": "IMP001", "text": "[guttural roar] Feed... the machine!"}, {"stem": "IMP001_death", "cast": "IMP001", "text": "[distorted, dying groan] Systems... failing..."}, {"stem": "IMP074_play", "cast": "IMP074", "text": "[guttural, labored] Stitched... together..."}, {"stem": "TOKEN_EXPERIMENT_play", "cast": "TOKEN_EXPERIMENT", "text": "[guttural] Obey... master!"}, {"stem": "IMP020_cast", "cast": "SOV_M", "text": "[cold, commanding] Out with the old. Let the mutation decide what stays."}, {"stem": "IMP020_castf", "cast": "SOV_F", "text": "[cold, commanding] Out with the old. Let the mutation decide what stays."}, {"stem": "IMP021_cast", "cast": "SOV_M", "text": "[shouting] Unleash them! Straight into their lines, and don't stop!"}, {"stem": "IMP021_castf", "cast": "SOV_F", "text": "[shouting] Unleash them! Straight into their lines, and don't stop!"}, {"stem": "IMP022_cast", "cast": "SOV_M", "text": "[cold] Exterminate. Leave nothing breathing."}, {"stem": "IMP022_castf", "cast": "SOV_F", "text": "[cold] Exterminate. Leave nothing breathing."}, {"stem": "IMP023_cast", "cast": "SOV_M", "text": "[grim] An eye for an eye. Your champion falls with mine."}, {"stem": "IMP023_castf", "cast": "SOV_F", "text": "[grim] An eye for an eye. Your champion falls with mine."}, {"stem": "IMP024_cast", "cast": "SOV_M", "text": "[dry, darkly amused] Sacrifice the subject. And do take notes."}, {"stem": "IMP024_castf", "cast": "SOV_F", "text": "[dry, darkly amused] Sacrifice the subject. And do take notes."}, {"stem": "IMP025_cast", "cast": "SOV_M", "text": "[smug] Not so fast! Vanish into the mist."}, {"stem": "IMP025_castf", "cast": "SOV_F", "text": "[smug] Not so fast! Vanish into the mist."}, {"stem": "IMP026_cast", "cast": "SOV_M", "text": "[intense, quiet] Now. Strike, and don't give them a second chance."}, {"stem": "IMP026_castf", "cast": "SOV_F", "text": "[intense, quiet] Now. Strike, and don't give them a second chance."}, {"stem": "IMP027_cast", "cast": "SOV_M", "text": "[contemptuous] Stamp out this little rebellion before it spreads."}, {"stem": "IMP027_castf", "cast": "SOV_F", "text": "[contemptuous] Stamp out this little rebellion before it spreads."}, {"stem": "IMP028_cast", "cast": "SOV_M", "text": "[casual, dark] Inject the serum. Ignore the side effects."}, {"stem": "IMP028_castf", "cast": "SOV_F", "text": "[casual, dark] Inject the serum. Ignore the side effects."}, {"stem": "IMP029_cast", "cast": "SOV_M", "text": "[amused] Strap it on. When they get close... boom."}, {"stem": "IMP029_castf", "cast": "SOV_F", "text": "[amused] Strap it on. When they get close... boom."}, {"stem": "IMP030_cast", "cast": "SOV_M", "text": "[thoughtful] Dig through the blight. Knowledge always costs something."}, {"stem": "IMP030_castf", "cast": "SOV_F", "text": "[thoughtful] Dig through the blight. Knowledge always costs something."}, {"stem": "IMP031_cast", "cast": "SOV_M", "text": "[sly] Welcome to the lab. Mind the vials."}, {"stem": "IMP031_castf", "cast": "SOV_F", "text": "[sly] Welcome to the lab. Mind the vials."}, {"stem": "IMP032_cast", "cast": "SOV_M", "text": "[conspiratorial whisper] Lay low. Somebody else is paying the rent."}, {"stem": "IMP032_castf", "cast": "SOV_F", "text": "[conspiratorial whisper] Lay low. Somebody else is paying the rent."}, {"stem": "IMP052_cast", "cast": "SOV_M", "text": "[commanding] Coordinates locked. Make it rain fire!"}, {"stem": "IMP052_castf", "cast": "SOV_F", "text": "[commanding] Coordinates locked. Make it rain fire!"}, {"stem": "IMP053_cast", "cast": "SOV_M", "text": "[cheerful] Everyone pitches in! Many hands, full supplies."}, {"stem": "IMP053_castf", "cast": "SOV_F", "text": "[cheerful] Everyone pitches in! Many hands, full supplies."}, {"stem": "IMP054_cast", "cast": "SOV_M", "text": "[urgent] Clear! Get back in the fight!"}, {"stem": "IMP054_castf", "cast": "SOV_F", "text": "[urgent] Clear! Get back in the fight!"}, {"stem": "IMP055_cast", "cast": "SOV_M", "text": "[shouting] Pop smoke! Let them shoot at shadows!"}, {"stem": "IMP055_castf", "cast": "SOV_F", "text": "[shouting] Pop smoke! Let them shoot at shadows!"}, {"stem": "IMP056_cast", "cast": "SOV_M", "text": "[stern, righteous] Begone, by order of the light."}, {"stem": "IMP056_castf", "cast": "SOV_F", "text": "[stern, righteous] Begone, by order of the light."}, {"stem": "IMP057_cast", "cast": "SOV_M", "text": "[urgent, straining] Hold on! Pulling you out now!"}, {"stem": "IMP057_castf", "cast": "SOV_F", "text": "[urgent, straining] Hold on! Pulling you out now!"}, {"stem": "IMP058_cast", "cast": "SOV_M", "text": "[rousing shout] On your feet, soldier! The war isn't waiting!"}, {"stem": "IMP058_castf", "cast": "SOV_F", "text": "[rousing shout] On your feet, soldier! The war isn't waiting!"}, {"stem": "IMP059_cast", "cast": "SOV_M", "text": "[calm, reassuring] Barrier online. Stay behind it."}, {"stem": "IMP059_castf", "cast": "SOV_F", "text": "[calm, reassuring] Barrier online. Stay behind it."}, {"stem": "IMP060_cast", "cast": "SOV_M", "text": "[wry] Suit up. You're the one they'll shoot at, so look good."}, {"stem": "IMP060_castf", "cast": "SOV_F", "text": "[wry] Suit up. You're the one they'll shoot at, so look good."}, {"stem": "IMP061_cast", "cast": "SOV_M", "text": "[smooth] Credit approved. Spend it like you mean it."}, {"stem": "IMP061_castf", "cast": "SOV_F", "text": "[smooth] Credit approved. Spend it like you mean it."}, {"stem": "IMP062_cast", "cast": "SOV_M", "text": "[reverent, soft] Sanctuary for the faithful fallen."}, {"stem": "IMP062_castf", "cast": "SOV_F", "text": "[reverent, soft] Sanctuary for the faithful fallen."}, {"stem": "IMP063_cast", "cast": "SOV_M", "text": "[stern, official] Quarantine is in effect. Nobody in, nobody out."}, {"stem": "IMP063_castf", "cast": "SOV_F", "text": "[stern, official] Quarantine is in effect. Nobody in, nobody out."}, {"stem": "IMP082_cast", "cast": "SOV_M", "text": "[excited] Supplies incoming from the clouds!"}, {"stem": "IMP082_castf", "cast": "SOV_F", "text": "[excited] Supplies incoming from the clouds!"}, {"stem": "IMP083_cast", "cast": "SOV_M", "text": "[ominous] Open the floodgates. Wash it all away."}, {"stem": "IMP083_castf", "cast": "SOV_F", "text": "[ominous] Open the floodgates. Wash it all away."}, {"stem": "IMP084_cast", "cast": "SOV_M", "text": "[quietly confident] The pieces are in motion. They just don't know it yet."}, {"stem": "IMP084_castf", "cast": "SOV_F", "text": "[quietly confident] The pieces are in motion. They just don't know it yet."}, {"stem": "IMP085_cast", "cast": "SOV_M", "text": "[menacing, calm] Talk. The pain stops when the truth starts."}, {"stem": "IMP085_castf", "cast": "SOV_F", "text": "[menacing, calm] Talk. The pain stops when the truth starts."}, {"stem": "IMP086_cast", "cast": "SOV_M", "text": "[forceful] Back where you came from, and stay there!"}, {"stem": "IMP086_castf", "cast": "SOV_F", "text": "[forceful] Back where you came from, and stay there!"}, {"stem": "IMP087_cast", "cast": "SOV_M", "text": "[panicked, hurried] Pack it up! Save the specimen!"}, {"stem": "IMP087_castf", "cast": "SOV_F", "text": "[panicked, hurried] Pack it up! Save the specimen!"}, {"stem": "IMP088_cast", "cast": "SOV_M", "text": "[gleeful] Into the void with you!"}, {"stem": "IMP088_castf", "cast": "SOV_F", "text": "[gleeful] Into the void with you!"}, {"stem": "IMP089_cast", "cast": "SOV_M", "text": "[sly] Scramble their thoughts. Let them forget their own plan."}, {"stem": "IMP089_castf", "cast": "SOV_F", "text": "[sly] Scramble their thoughts. Let them forget their own plan."}, {"stem": "IMP090_cast", "cast": "SOV_M", "text": "[warm, wistful] A lucky charm for wanderers."}, {"stem": "IMP090_castf", "cast": "SOV_F", "text": "[warm, wistful] A lucky charm for wanderers."}, {"stem": "IMP091_cast", "cast": "SOV_M", "text": "[businesslike] Cargo secured. Cash it in later."}, {"stem": "IMP091_castf", "cast": "SOV_F", "text": "[businesslike] Cargo secured. Cash it in later."}, {"stem": "IMP092_cast", "cast": "SOV_M", "text": "[curious, hushed] Lost secrets, waiting to be read."}, {"stem": "IMP092_castf", "cast": "SOV_F", "text": "[curious, hushed] Lost secrets, waiting to be read."}, {"stem": "IMP093_cast", "cast": "SOV_M", "text": "[smug] Jamming tower online. Enjoy the static."}, {"stem": "IMP093_castf", "cast": "SOV_F", "text": "[smug] Jamming tower online. Enjoy the static."}, {"stem": "IMP094_cast", "cast": "SOV_M", "text": "[alert] Radar sweep active. Stay alert."}, {"stem": "IMP094_castf", "cast": "SOV_F", "text": "[alert] Radar sweep active. Stay alert."}, {"stem": "IMP095_cast", "cast": "SOV_M", "text": "[exhilarated shout] Ride the storm! It never stops!"}, {"stem": "IMP095_castf", "cast": "SOV_F", "text": "[exhilarated shout] Ride the storm! It never stops!"}, {"stem": "IMP115_cast", "cast": "SOV_M", "text": "[cruel, possessive] Brand them. Everyone should know who they belong to."}, {"stem": "IMP115_castf", "cast": "SOV_F", "text": "[cruel, possessive] Brand them. Everyone should know who they belong to."}, {"stem": "IMP116_cast", "cast": "SOV_M", "text": "[alarmed shout] The mountain awakens! Run!"}, {"stem": "IMP116_castf", "cast": "SOV_F", "text": "[alarmed shout] The mountain awakens! Run!"}, {"stem": "IMP117_cast", "cast": "SOV_M", "text": "[quiet threat] One more scratch, and you die."}, {"stem": "IMP117_castf", "cast": "SOV_F", "text": "[quiet threat] One more scratch, and you die."}, {"stem": "IMP118_cast", "cast": "SOV_M", "text": "[wild shout] Smash it! Take everything!"}, {"stem": "IMP118_castf", "cast": "SOV_F", "text": "[wild shout] Smash it! Take everything!"}, {"stem": "IMP119_cast", "cast": "SOV_M", "text": "[excited shout] Light the fuses! Rockets away!"}, {"stem": "IMP119_castf", "cast": "SOV_F", "text": "[excited shout] Light the fuses! Rockets away!"}, {"stem": "IMP120_cast", "cast": "SOV_M", "text": "[rousing] Rise up and strike them down!"}, {"stem": "IMP120_castf", "cast": "SOV_F", "text": "[rousing] Rise up and strike them down!"}, {"stem": "IMP121_cast", "cast": "SOV_M", "text": "[fierce] Get angry. Then get back up!"}, {"stem": "IMP121_castf", "cast": "SOV_F", "text": "[fierce] Get angry. Then get back up!"}, {"stem": "IMP122_cast", "cast": "SOV_M", "text": "[proud] Every kill, a song of glory."}, {"stem": "IMP122_castf", "cast": "SOV_F", "text": "[proud] Every kill, a song of glory."}, {"stem": "IMP123_cast", "cast": "SOV_M", "text": "[grim, solemn] Fall, and vengeance returns."}, {"stem": "IMP123_castf", "cast": "SOV_F", "text": "[grim, solemn] Fall, and vengeance returns."}, {"stem": "IMP124_cast", "cast": "SOV_M", "text": "[rallying shout] Pack, gather here! We rise together!"}, {"stem": "IMP124_castf", "cast": "SOV_F", "text": "[rallying shout] Pack, gather here! We rise together!"}, {"stem": "IMP125_cast", "cast": "SOV_M", "text": "[eager] Start digging. Treasure lies below."}, {"stem": "IMP125_castf", "cast": "SOV_F", "text": "[eager] Start digging. Treasure lies below."}, {"stem": "IMP126_cast", "cast": "SOV_M", "text": "[warning] Mind the cliffs. Nobody survives the fall."}, {"stem": "IMP126_castf", "cast": "SOV_F", "text": "[warning] Mind the cliffs. Nobody survives the fall."}, {"stem": "IMP145_cast", "cast": "SOV_M", "text": "[tense, then shouting] Together, on my signal. Now!"}, {"stem": "IMP145_castf", "cast": "SOV_F", "text": "[tense, then shouting] Together, on my signal. Now!"}, {"stem": "IMP146_cast", "cast": "SOV_M", "text": "[predatory whisper] The prey is tired. Pounce!"}, {"stem": "IMP146_castf", "cast": "SOV_F", "text": "[predatory whisper] The prey is tired. Pounce!"}, {"stem": "IMP147_cast", "cast": "SOV_M", "text": "[fierce] Let the wild roar silence them!"}, {"stem": "IMP147_castf", "cast": "SOV_F", "text": "[fierce] Let the wild roar silence them!"}, {"stem": "IMP148_cast", "cast": "SOV_M", "text": "[soft, hypnotic] Breathe deep... and sleep."}, {"stem": "IMP148_castf", "cast": "SOV_F", "text": "[soft, hypnotic] Breathe deep... and sleep."}, {"stem": "IMP149_cast", "cast": "SOV_M", "text": "[gleeful shout] Surprise! Ambush from the treetops!"}, {"stem": "IMP149_castf", "cast": "SOV_F", "text": "[gleeful shout] Surprise! Ambush from the treetops!"}, {"stem": "IMP150_cast", "cast": "SOV_M", "text": "[defiant] We hold here, to the last!"}, {"stem": "IMP150_castf", "cast": "SOV_F", "text": "[defiant] We hold here, to the last!"}, {"stem": "IMP151_cast", "cast": "SOV_M", "text": "[playful] Up the trees, out of reach!"}, {"stem": "IMP151_castf", "cast": "SOV_F", "text": "[playful] Up the trees, out of reach!"}, {"stem": "IMP152_cast", "cast": "SOV_M", "text": "[mischievous] Scatter the spikes. Let them dance."}, {"stem": "IMP152_castf", "cast": "SOV_F", "text": "[mischievous] Scatter the spikes. Let them dance."}, {"stem": "IMP153_cast", "cast": "SOV_M", "text": "[serene] Life ends. Life returns."}, {"stem": "IMP153_castf", "cast": "SOV_F", "text": "[serene] Life ends. Life returns."}, {"stem": "IMP154_cast", "cast": "SOV_M", "text": "[nostalgic, soft] An heirloom of moonlit days."}, {"stem": "IMP154_castf", "cast": "SOV_F", "text": "[nostalgic, soft] An heirloom of moonlit days."}, {"stem": "IMP155_cast", "cast": "SOV_M", "text": "[solemn, rousing] Ancestors of Elorhain, defend your children!"}, {"stem": "IMP155_castf", "cast": "SOV_F", "text": "[solemn, rousing] Ancestors of Elorhain, defend your children!"}, {"stem": "IMP156_cast", "cast": "SOV_M", "text": "[calm, ominous] Let the vines bind them."}, {"stem": "IMP156_castf", "cast": "SOV_F", "text": "[calm, ominous] Let the vines bind them."}, {"stem": "IMP157_cast", "cast": "SOV_M", "text": "[wry] Always keep a way out."}, {"stem": "IMP157_castf", "cast": "SOV_F", "text": "[wry] Always keep a way out."}, {"stem": "IMP158_cast", "cast": "SOV_M", "text": "[quiet awe] Nature reclaims the machine."}, {"stem": "IMP158_castf", "cast": "SOV_F", "text": "[quiet awe] Nature reclaims the machine."}, {"stem": "IMP161_cast", "cast": "SOV_M", "text": "[commanding] Dig in behind the concrete!"}, {"stem": "IMP161_castf", "cast": "SOV_F", "text": "[commanding] Dig in behind the concrete!"}, {"stem": "IMP164_cast", "cast": "SOV_M", "text": "[rousing shout] Raise the banner! Reinforcements, come!"}, {"stem": "IMP164_castf", "cast": "SOV_F", "text": "[rousing shout] Raise the banner! Reinforcements, come!"}, {"stem": "IMP165_cast", "cast": "SOV_M", "text": "[cold, quiet] Mines armed. One wrong step, and it's over."}, {"stem": "IMP165_castf", "cast": "SOV_F", "text": "[cold, quiet] Mines armed. One wrong step, and it's over."}, {"stem": "ANN_yourTurn", "cast": "ANN", "text": "[confident] Your move, Sovereign."}, {"stem": "ANN_victory", "cast": "ANN", "text": "[triumphant] Victory! Elyrion bows to you."}, {"stem": "ANN_defeat", "cast": "ANN", "text": "[somber] Defeat... your banners fall."}, {"stem": "ANN_begin", "cast": "ANN", "text": "[dramatic] Take your positions. The battle begins!"}, {"stem": "ANN_zoneWon", "cast": "ANN", "text": "[excited] Zone secured!"}]}""")
JOBS = BIBLE['jobs']
TOTAL = sum(len(j['text']) for j in JOBS)
PREMADE = [v for v in VOICES if v.get('category') == 'premade']
BY_NAME = {}
for v in PREMADE: BY_NAME.setdefault(v['name'].split(' - ')[0].strip(), v)
MINE = {v['name']: v for v in VOICES if v['name'].startswith('EoA ')}
used = SUB.get('voice_slots_used')
if used is None: used = sum(1 for v in VOICES if v.get('category') in ('generated', 'cloned') and not v['name'].startswith('EoA '))
limit = SUB.get('voice_limit') or 10
FREE = max(0, limit - used - len(MINE))
left = (SUB.get('character_limit', 30000) - SUB.get('character_count', 0)) if SUB else 30000
n_new = min(FREE, len([d for d in BIBLE['designs'] if 'EoA ' + d['key'] not in MINE]))
design_cost = n_new * 600
TAKES = [0.5, 0.0] if left >= 2.15 * TOTAL + design_cost else [0.5]
print(f'Clips: {len(JOBS)} · characters per take: {TOTAL:,}')
print(f'Free voice slots: {FREE} (EoA voices already on the account: {len(MINE)}) → new designed voices: {n_new}')
print(f'Takes per line: {len(TAKES)} ({"Natural + Creative" if len(TAKES) == 2 else "Natural"}); estimated cost ≈ {len(TAKES) * TOTAL + design_cost:,} credits of {left:,} available')
if left < TOTAL + design_cost:
    print('WARNING: not enough credits for everything. Step 5 will record as much as it can; finish the rest next month or after topping up.')


In [ ]:
#@title 3) Design unique voices (listen to them)
DESIGNED = {v['name'][4:]: v['voice_id'] for v in MINE.values()}
free = FREE
for d in BIBLE['designs']:
    key = d['key']
    if key in DESIGNED:
        print(f'✓ {key}: already on the account'); continue
    if free <= 0:
        print(f'– {key}: no free slot, a default voice will be used'); continue
    r = call('POST', '/v1/text-to-voice/design', json={'voice_description': d['description'], 'model_id': 'eleven_ttv_v3', 'text': d['text'], 'seed': d['seed']})
    if r.status_code != 200:
        r = call('POST', '/v1/text-to-voice/design', json={'voice_description': d['description'], 'text': d['text'], 'seed': d['seed']})
    if r.status_code != 200:
        print(f'✗ {key}: design failed ({r.status_code}) {err(r)}'); continue
    pv = r.json()['previews'][0]
    body = {'voice_name': 'EoA ' + key, 'voice_description': d['description'], 'generated_voice_id': pv['generated_voice_id']}
    c = call('POST', '/v1/text-to-voice', json=body)
    if c.status_code in (404, 405):
        c = call('POST', '/v1/text-to-voice/create-voice-from-preview', json=body)
    if c.status_code != 200:
        print(f'✗ {key}: could not save the voice ({c.status_code}) {err(c)}'); continue
    DESIGNED[key] = c.json()['voice_id']; free -= 1
    player(base64.b64decode(pv['audio_base_64']), pv.get('media_type') or 'audio/mpeg', f'{key}: {d["description"][:90]}…')
print(f'Designed voices ready: {len(DESIGNED)}')


In [ ]:
#@title 4) Cast and a short test (4 lines)
def resolve(key):
    c = BIBLE['cast'][key]
    if c['design'] and c['design'] in DESIGNED: return DESIGNED[c['design']], 'EoA ' + c['design']
    for n in c['premade']:
        if n in BY_NAME: return BY_NAME[n]['voice_id'], n
    for v in PREMADE:
        if (v.get('labels') or {}).get('gender') == c['gender']: return v['voice_id'], v['name']
    v = (PREMADE or VOICES)[0]; return v['voice_id'], v['name']
CASTMAP = {k: resolve(k) for k in BIBLE['cast']}
FORMAT = 'mp3_44100_64'

def tts(voice_id, text, stability):
    global FORMAT
    body = {'text': text, 'model_id': 'eleven_v3', 'voice_settings': {'stability': stability, 'similarity_boost': 0.75}}
    r = call('POST', f'/v1/text-to-speech/{voice_id}?output_format={FORMAT}', json=body)
    if r.status_code in (400, 422) and 'output_format' in r.text:
        FORMAT = 'mp3_44100_128'; r = call('POST', f'/v1/text-to-speech/{voice_id}?output_format={FORMAT}', json=body)
    if r.status_code in (400, 422):
        r = call('POST', f'/v1/text-to-speech/{voice_id}?output_format={FORMAT}', json={'text': text, 'model_id': 'eleven_v3'})
    if r.status_code != 200: raise RuntimeError(f'{r.status_code}: {err(r)}')
    return r.content

roles = {}
for k, (vid, name) in CASTMAP.items(): roles.setdefault(name, []).append(k)
print('Cast (voice → roles):')
for name, ks in sorted(roles.items()): print(f'  {name}: {", ".join(ks)}')
by_stem = {j['stem']: j for j in JOBS}
for stem in ['IMP015_play', 'IMP003_play', 'IMP111_play', 'IMP051_play']:
    j = by_stem[stem]
    try:
        player(tts(CASTMAP[j['cast']][0], j['text'], 0.5), 'audio/mpeg', f'{stem} ({CASTMAP[j["cast"]][1]}): {j["text"]}')
    except Exception as e:
        print('Test error:', stem, e)


In [ ]:
#@title 5) Record everything (you can run it again: it continues where it stopped)
OUTD = '/content/eoa_voices'; os.makedirs(OUTD, exist_ok=True)
todo = [(j, k, st) for j in JOBS for k, st in enumerate(TAKES)]
todo = [t for t in todo if not (os.path.exists(f"{OUTD}/{t[0]['stem']}__t{t[1]}.mp3") and os.path.getsize(f"{OUTD}/{t[0]['stem']}__t{t[1]}.mp3") > 800)]
print(f'To record: {len(todo)} clips')
lock = threading.Lock(); done = [0]; fails = []; t0 = time.time(); stop = [False]

def work(item):
    j, k, st = item
    if stop[0]: return
    try:
        audio = tts(CASTMAP[j['cast']][0], j['text'], st)
        with open(f"{OUTD}/{j['stem']}__t{k}.mp3", 'wb') as f: f.write(audio)
    except Exception as e:
        with lock: fails.append((j['stem'], k, str(e)[:200]))
        if 'quota' in str(e).lower() or 'credits' in str(e).lower(): stop[0] = True
    with lock:
        done[0] += 1
        if done[0] % 20 == 0 or done[0] == len(todo):
            el = time.time() - t0; eta = el / done[0] * (len(todo) - done[0])
            print(f'{done[0]}/{len(todo)} · {el / 60:.1f} min · about {eta / 60:.0f} min left', flush=True)

with cf.ThreadPoolExecutor(max_workers=2) as ex: list(ex.map(work, todo))
json.dump({'cast': {k: {'voice_id': v[0], 'voice': v[1]} for k, v in CASTMAP.items()}, 'takes': TAKES, 'format': FORMAT, 'jobs': JOBS},
          open(f'{OUTD}/manifest.json', 'w'), ensure_ascii=False, indent=1)
print(f'Done. Errors: {len(fails)}')
for f in fails[:15]: print('  ', f)
if stop[0]: print('Recording stopped: credits ran out. Top up or wait for the reset, then run step 5 again.')
elif fails: print('Run step 5 again to retry the failed clips.')


In [ ]:
#@title 6) Pack into ZIP and download
from google.colab import files
names = sorted(n for n in os.listdir(OUTD) if n.endswith('.mp3'))
parts, cur, size = [], [], 0
for n in names:
    s = os.path.getsize(f'{OUTD}/{n}')
    if cur and size + s > 22_000_000: parts.append(cur); cur, size = [], 0
    cur.append(n); size += s
if cur: parts.append(cur)
for i, p in enumerate(parts, 1):
    zp = f'/content/eoa_voices_part{i}.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_STORED) as z:
        if i == 1: z.write(f'{OUTD}/manifest.json', 'manifest.json')
        for n in p: z.write(f'{OUTD}/{n}', n)
    print(f'Part {i}: {len(p)} files, {os.path.getsize(zp) / 1e6:.1f} MB')
    files.download(zp)
print('If the browser asks whether to allow multiple downloads, choose Allow. Send all the parts to Claude.')


In [ ]:
#@title 7) (optional) Remove the designed voices from your account
mine = [v for v in call('GET', '/v1/voices').json().get('voices', []) if v['name'].startswith('EoA ')]
for v in mine:
    r = call('DELETE', f"/v1/voices/{v['voice_id']}")
    print(('✓ removed ' if r.status_code == 200 else f'✗ {r.status_code} ') + v['name'])
print('Done. The recorded clips are not affected.')
